# 04 — Ревизия: дедупликация, честные split'ы, 3 seed'а, доверительные интервалы, калибровка порога

Отвечает на замечания рецензента к v2:
1. **Дубликаты и «родственные» изображения между split'ами.** Считаем перцептивный хеш (dHash) всех изображений,
   объединяем почти одинаковые картинки одного генератора в кластеры и заново делим CF-часть так, чтобы
   целый кластер попадал только в один split. Отложенные генераторы (test_unseen) и наш набор (test_own) не меняются.
2. **Повторные прогоны.** SimpleCNN и ResNet-50 обучаются с тремя seed'ами; CLIP + LR детерминирован (одна подгонка).
3. **Неопределённость.** Среднее ± std по seed'ам и 95% bootstrap-интервалы по тестовым изображениям для всех метрик.
4. **Ранжирование vs порог.** На test_own отдельно: AUC, accuracy при пороге 0,5, при пороге, подобранном на val (CF),
   и при пороге, подобранном на 200 калибровочных изображениях из целевого домена (оценка на остальных 1 800).

Вход: `dataset.zip` (ноутбук 02) в Drive `deepfake_paper/`. Выход: `results_v2.zip` там же.
Ноутбук **резюмируемый**: результаты каждой модели и seed'а сразу сохраняются в Drive, повторный запуск пропускает готовое.
На T4 ≈ 1,5 часа; на CPU ≈ 8 часов (можно оставить на ночь).

In [ ]:
!pip -q install scikit-learn transformers

In [ ]:
import os, io, json, time, random, zipfile, shutil, collections
import numpy as np, pandas as pd
import torch, torch.nn as nn
import torchvision.transforms as T
from torchvision import models
from PIL import Image
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, balanced_accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

CFG = dict(
    DATASET_ZIP = ["/content/drive/MyDrive/deepfake_paper/dataset.zip", "/content/dataset.zip"],
    ROOT = "/content/dataset", RES = "/content/results_v2",
    DRIVE = "/content/drive/MyDrive/deepfake_paper/results_v2_partial",   # сюда падают промежуточные результаты
    SEEDS = [0, 1, 2],
    SPLIT_SEED = 42, HASH_THR = 8,          # hamming-порог dHash (64 бит) для «почти одинаковых» картинок
    IMG = 224, BS = 32, WORKERS = 4,
    EPOCHS_CNN = 10, LR_CNN = 1e-3, EPOCHS_RESNET = 5, LR_RESNET = 1e-4,
    CLIP_ID = "openai/clip-vit-large-patch14",
    N_BOOT = 1000, N_CAL = 200, N_CAL_REPEATS = 20,
)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
if DEV != "cuda":
    CFG.update(CLIP_ID="openai/clip-vit-base-patch16", WORKERS=2)
    torch.set_num_threads(os.cpu_count() or 2)
    print("GPU нет — режим CPU: CLIP ViT-B/16, те же эпохи; 3 seed'а займут ~8 часов")
CLIP_DTYPE = torch.float16 if DEV == "cuda" else torch.float32
print("device:", DEV, "| CLIP:", CFG["CLIP_ID"])
for d in (CFG["RES"], f"{CFG['RES']}/figs", f"{CFG['RES']}/splits_v2"):
    os.makedirs(d, exist_ok=True)
try:
    from google.colab import drive; drive.mount("/content/drive")
    os.makedirs(CFG["DRIVE"], exist_ok=True)
except Exception as e:
    print("Drive не подключён:", e)

## 1. Данные, перцептивные хеши, кластеры похожих изображений

In [ ]:
zpath = next((p for p in CFG["DATASET_ZIP"] if os.path.exists(p)), None)
assert zpath, "dataset.zip не найден"
if not os.path.exists(f"{CFG['ROOT']}/metadata.csv"):
    with zipfile.ZipFile(zpath) as z: z.extractall(CFG["ROOT"])
meta = pd.read_csv(f"{CFG['ROOT']}/metadata.csv", dtype=str).fillna("")
meta["y"] = (meta.label == "fake").astype(int)
meta["path"] = CFG["ROOT"] + "/images/" + meta.filename
meta.loc[(meta.origin == "cf") & (meta.y == 1), "fake_type"] = "generated"
meta["held_out"] = pd.to_numeric(meta.held_out, errors="coerce").fillna(0).astype(int)
meta["group"] = np.where(meta.y == 1, meta.generator, "real:" + meta.source)   # генератор или источник real
meta = meta.reset_index(drop=True)

def dhash(p, s=8):
    im = Image.open(p).convert("L").resize((s + 1, s), Image.LANCZOS)
    a = np.asarray(im, dtype=np.int16)
    return (a[:, 1:] > a[:, :-1]).ravel().astype(np.uint8)
bits = np.stack([dhash(p) for p in tqdm(meta.path, desc="dHash")]).astype(np.int32)
D = 64 - (bits @ bits.T + (1 - bits) @ (1 - bits).T); np.fill_diagonal(D, 99)

# кластеры: union-find по парам с hamming <= HASH_THR внутри одного генератора/источника (только CF-часть)
parent = list(range(len(meta)))
def find(i):
    while parent[i] != i: parent[i] = parent[parent[i]]; i = parent[i]
    return i
cf_idx = np.where(meta.origin == "cf")[0]
grp = meta.group.values
for i, j in np.argwhere(np.triu(D <= CFG["HASH_THR"])):
    if meta.origin[i] == "cf" and meta.origin[j] == "cf" and grp[i] == grp[j]:
        parent[find(i)] = find(j)
meta["cluster"] = [find(i) for i in range(len(meta))]
cs = meta[meta.origin == "cf"].groupby("cluster").size()
old_split = meta.split.copy()
old_cross = sum(1 for i, j in np.argwhere(np.triu(D <= CFG["HASH_THR"]))
                if meta.origin[i] == "cf" and meta.origin[j] == "cf" and old_split[i] != old_split[j])
report = dict(
    hash_threshold=CFG["HASH_THR"], cf_images=int(len(cf_idx)), cf_clusters=int(len(cs)),
    cf_images_in_multi_clusters=int(cs[cs > 1].sum()), largest_cluster=int(cs.max()),
    exact_duplicate_pairs=int(sum(1 for i, j in np.argwhere(np.triu(D == 0)) if meta.origin[i] == "cf" and meta.origin[j] == "cf")),
    v1_cross_split_near_duplicate_pairs=int(old_cross),
    multi_cluster_images_by_group={g: int(v) for g, v in meta[(meta.origin == "cf")].groupby("group").apply(lambda d: d.groupby("cluster").size().pipe(lambda s: s[s > 1].sum())).items() if v > 0},
)
print(json.dumps(report, indent=1, ensure_ascii=False))

## 2. Новые split'ы: целый кластер — в один split
Внутри каждой группы (генератор или real) кластеры перемешиваются с фиксированным seed и жадно раскладываются в train/val/test_seen
в пропорции 70/15/15 по числу изображений. Отложенные генераторы целиком остаются в test_unseen, наш набор — в test_own.

In [ ]:
rng = np.random.default_rng(CFG["SPLIT_SEED"])
meta["split_v2"] = meta.split
cf = meta[(meta.origin == "cf") & (meta.held_out == 0)]
for g, d in cf.groupby("group"):
    clusters = d.groupby("cluster").size().sample(frac=1, random_state=CFG["SPLIT_SEED"])
    n = len(d); targets = {"train": 0.70 * n, "val": 0.15 * n, "test_seen": 0.15 * n}; filled = {k: 0 for k in targets}
    for c, size in clusters.items():
        # кладём кластер туда, где сильнее всего недобор относительно цели
        k = max(targets, key=lambda k: (targets[k] - filled[k]) / targets[k])
        meta.loc[(meta.cluster == c) & (meta.group == g), "split_v2"] = k; filled[k] += size
meta.loc[(meta.origin == "cf") & (meta.held_out == 1), "split_v2"] = "test_unseen"
meta.loc[meta.origin == "own", "split_v2"] = "test_own"

new_cross = sum(1 for i, j in np.argwhere(np.triu(D <= CFG["HASH_THR"]))
                if meta.origin[i] == "cf" and meta.origin[j] == "cf" and meta.split_v2[i] != meta.split_v2[j])
report["v2_cross_split_near_duplicate_pairs"] = int(new_cross)
moved = int((meta.split != meta.split_v2).sum()); report["images_moved_between_splits"] = moved
print("пар похожих картинок через границу split'ов: было", old_cross, "→ стало", new_cross, "| переложено изображений:", moved)
print(meta.groupby(["split_v2", "label"]).size().unstack(fill_value=0))
json.dump(report, open(f"{CFG['RES']}/dedup_report.json", "w"), indent=1, ensure_ascii=False)
meta.drop(columns=["path"]).to_csv(f"{CFG['RES']}/metadata_v2.csv", index=False)
for s, d in meta.groupby("split_v2"):
    d.drop(columns=["path"]).to_csv(f"{CFG['RES']}/splits_v2/{s}.csv", index=False)
shutil.copy(f"{CFG['RES']}/dedup_report.json", CFG["DRIVE"]) if os.path.isdir(CFG["DRIVE"]) else None

train_df = meta[meta.split_v2 == "train"].reset_index(drop=True)
val_df   = meta[meta.split_v2 == "val"].reset_index(drop=True)
EVAL = {
    "test_seen":   meta[meta.split_v2 == "test_seen"].reset_index(drop=True),
    "test_unseen": pd.concat([meta[meta.split_v2 == "test_unseen"], meta[(meta.split_v2 == "test_seen") & (meta.y == 0)]]).reset_index(drop=True),
    "test_own":    meta[meta.split_v2 == "test_own"].reset_index(drop=True),
}
print("train:", len(train_df), "val:", len(val_df), {k: len(v) for k, v in EVAL.items()})

## 3. Общие функции (как в ноутбуке 03) + сохранение по ходу

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = T.Compose([T.Resize(256), T.RandomCrop(CFG["IMG"]), T.RandomHorizontalFlip(), T.ToTensor(), T.Normalize(MEAN, STD)])
eval_tf  = T.Compose([T.Resize(256), T.CenterCrop(CFG["IMG"]), T.ToTensor(), T.Normalize(MEAN, STD)])
class ImgDS(torch.utils.data.Dataset):
    def __init__(self, df, tf): self.p, self.y, self.tf = df.path.tolist(), df.y.values.astype(np.float32), tf
    def __len__(self): return len(self.p)
    def __getitem__(self, i): return self.tf(Image.open(self.p[i]).convert("RGB")), self.y[i]
def loader(df, tf, shuffle=False, seed=0):
    g = torch.Generator(); g.manual_seed(seed)
    return torch.utils.data.DataLoader(ImgDS(df, tf), batch_size=CFG["BS"], shuffle=shuffle, generator=g,
                                       num_workers=CFG["WORKERS"], pin_memory=(DEV == "cuda"), persistent_workers=True)
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if DEV == "cuda": torch.cuda.manual_seed_all(s)

def metrics(y, p, thr=0.5):
    y, p = np.asarray(y), np.asarray(p); pred = (p >= thr).astype(int)
    out = dict(n=len(y), acc=accuracy_score(y, pred), bacc=balanced_accuracy_score(y, pred) if len(set(y)) == 2 else np.nan)
    out["auc"] = roc_auc_score(y, p) if len(set(y)) == 2 else np.nan
    out["ap"]  = average_precision_score(y, p) if len(set(y)) == 2 else np.nan
    out["tpr"] = pred[y == 1].mean() if (y == 1).any() else np.nan
    out["tnr"] = 1 - pred[y == 0].mean() if (y == 0).any() else np.nan
    return out

@torch.no_grad()
def predict(model, df):
    model.eval(); out = []
    for x, _ in loader(df, eval_tf):
        with torch.autocast(DEV, enabled=(DEV == "cuda")):
            out.append(torch.sigmoid(model(x.to(DEV)).float().squeeze(1)).cpu())
    return torch.cat(out).numpy()

def train(model, name, epochs, lr, seed):
    set_seed(seed); model.to(DEV)
    pos_w = torch.tensor([(train_df.y == 0).sum() / max((train_df.y == 1).sum(), 1)], device=DEV)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scaler = torch.amp.GradScaler(DEV, enabled=(DEV == "cuda"))
    tl = loader(train_df, train_tf, shuffle=True, seed=seed)
    best_auc, best_state, log = -1, None, []
    for ep in range(epochs):
        model.train(); tot = 0; t0 = time.time()
        for x, y in tqdm(tl, desc=f"{name} s{seed} ep{ep+1}/{epochs}", leave=False):
            x, y = x.to(DEV), y.to(DEV); opt.zero_grad(set_to_none=True)
            with torch.autocast(DEV, enabled=(DEV == "cuda")):
                loss = crit(model(x).squeeze(1).float(), y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); tot += loss.item() * len(y)
        sched.step()
        vm = metrics(val_df.y, predict(model, val_df))
        log.append(dict(seed=seed, epoch=ep + 1, train_loss=tot / len(train_df), val_auc=vm["auc"], val_acc=vm["acc"], sec=time.time() - t0))
        print(f"{name} seed{seed} ep{ep+1}: loss={tot/len(train_df):.4f} val_auc={vm['auc']:.4f} val_acc={vm['acc']:.4f} ({time.time()-t0:.0f}s)")
        if vm["auc"] > best_auc: best_auc, best_state = vm["auc"], {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    pd.DataFrame(log).to_csv(f"{CFG['RES']}/trainlog_{name}_seed{seed}.csv", index=False)
    return model

PRED = collections.defaultdict(dict)   # PRED[model][seed] = {"val":..., "test_seen":..., ...}
def pred_file(name, seed): return f"pred_{name}_seed{seed}.npz"
def save_pred(name, seed):
    f = pred_file(name, seed); np.savez(f"{CFG['RES']}/{f}", **PRED[name][seed])
    if os.path.isdir(CFG["DRIVE"]): shutil.copy(f"{CFG['RES']}/{f}", CFG["DRIVE"])
def load_pred(name, seed):
    for p in (f"{CFG['RES']}/{pred_file(name, seed)}", f"{CFG['DRIVE']}/{pred_file(name, seed)}"):
        if os.path.exists(p):
            z = np.load(p); PRED[name][seed] = {k: z[k] for k in z.files}; print(f"{name} seed{seed}: загружено из {p}"); return True
    return False
def run_and_store(model, name, seed):
    PRED[name][seed] = {"val": predict(model, val_df), **{k: predict(model, v) for k, v in EVAL.items()}}
    save_pred(name, seed)

## 4. SimpleCNN и ResNet-50 × 3 seed'а (резюмируемо)

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        def block(i, o): return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(True), nn.MaxPool2d(2))
        self.f = nn.Sequential(block(3, 32), block(32, 64), block(64, 128), block(128, 256), nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.h = nn.Sequential(nn.Dropout(0.3), nn.Linear(256, 1))
    def forward(self, x): return self.h(self.f(x))

for seed in CFG["SEEDS"]:
    if not load_pred("SimpleCNN", seed):
        set_seed(seed); m = train(SimpleCNN(), "simple_cnn", CFG["EPOCHS_CNN"], CFG["LR_CNN"], seed)
        run_and_store(m, "SimpleCNN", seed); del m; torch.cuda.empty_cache()
    if not load_pred("ResNet-50", seed):
        set_seed(seed); m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2); m.fc = nn.Linear(m.fc.in_features, 1)
        m = train(m, "resnet50", CFG["EPOCHS_RESNET"], CFG["LR_RESNET"], seed)
        run_and_store(m, "ResNet-50", seed); del m; torch.cuda.empty_cache()
print("сети готовы:", {k: sorted(v) for k, v in PRED.items()})

## 5. CLIP + логистическая регрессия (признаки кэшируются по id изображения)

In [ ]:
feat_path = f"{CFG['RES']}/clip_feats_all.npz"; feat_drive = f"{CFG['DRIVE']}/clip_feats_all_{CFG['CLIP_ID'].split('/')[-1]}.npz"
if not os.path.exists(feat_path) and os.path.exists(feat_drive): shutil.copy(feat_drive, feat_path)
if os.path.exists(feat_path):
    z = np.load(feat_path, allow_pickle=True); FEAT = dict(zip(z["ids"], z["feats"]))
else:
    from transformers import CLIPModel, CLIPImageProcessor
    clip = CLIPModel.from_pretrained(CFG["CLIP_ID"], torch_dtype=CLIP_DTYPE).to(DEV).eval()
    proc = CLIPImageProcessor.from_pretrained(CFG["CLIP_ID"])
    feats = []
    with torch.no_grad():
        for i in tqdm(range(0, len(meta), CFG["BS"]), desc="CLIP feats"):
            ims = [Image.open(p).convert("RGB").resize((256, 256)) for p in meta.path.iloc[i:i + CFG["BS"]]]
            px = proc(images=ims, return_tensors="pt")["pixel_values"].to(DEV, CLIP_DTYPE)
            out = clip.get_image_features(pixel_values=px)
            f = out if torch.is_tensor(out) else (out.image_embeds if getattr(out, "image_embeds", None) is not None else out.pooler_output)
            feats.append(torch.nn.functional.normalize(f.float(), dim=-1).cpu().numpy())
    feats = np.concatenate(feats); np.savez(feat_path, ids=meta.id.values, feats=feats)
    if os.path.isdir(CFG["DRIVE"]): shutil.copy(feat_path, feat_drive)
    FEAT = dict(zip(meta.id.values, feats)); del clip; torch.cuda.empty_cache()
F = lambda df: np.stack([FEAT[i] for i in df.id])
sc = StandardScaler().fit(F(train_df)); best = None
for C in [0.01, 0.1, 1.0, 10.0]:
    lr_ = LogisticRegression(C=C, max_iter=3000, class_weight="balanced").fit(sc.transform(F(train_df)), train_df.y)
    auc = roc_auc_score(val_df.y, lr_.predict_proba(sc.transform(F(val_df)))[:, 1]); print(f"C={C}: val_auc={auc:.4f}")
    if best is None or auc > best[0]: best = (auc, C, lr_)
_, bestC, clf = best; print("выбрано C =", bestC)
PRED["CLIP+LR"][0] = {"val": clf.predict_proba(sc.transform(F(val_df)))[:, 1], **{k: clf.predict_proba(sc.transform(F(v)))[:, 1] for k, v in EVAL.items()}}
save_pred("CLIP+LR", 0)

## 6. Метрики: по seed'ам, среднее ± std, bootstrap 95% CI
CI: 1000 раз пересэмплируем тестовые изображения (с возвращением); для сетей метрика считается для каждого seed'а и усредняется.

In [ ]:
MODELS = ["SimpleCNN", "ResNet-50", "CLIP+LR"]
rows = []
for m in MODELS:
    for s, d in PRED[m].items():
        for t, p in d.items():
            if t == "val": continue
            rows.append(dict(model=m, seed=s, test=t, **metrics(EVAL[t].y, p)))
            if t == "test_own":
                for ft in ("t2i", "i2i"):
                    mk = (EVAL[t].fake_type == ft).values
                    rows.append(dict(model=m, seed=s, test=f"test_own/{ft}", n=int(mk.sum()), tpr=float((p[mk] >= 0.5).mean())))
per_seed = pd.DataFrame(rows); per_seed.to_csv(f"{CFG['RES']}/metrics_per_seed.csv", index=False)

rng = np.random.default_rng(0)
KEYS = ("acc", "bacc", "auc", "ap", "tpr", "tnr")
def boot_ci(m, t):
    """95% percentile-интервалы для всех метрик сразу: пересэмплируем тестовые изображения, усредняем по seed'ам."""
    y = EVAL[t].y.values; P = [PRED[m][s][t] for s in PRED[m]]; vals = {k: [] for k in KEYS}
    for _ in range(CFG["N_BOOT"]):
        idx = rng.integers(0, len(y), len(y))
        if len(set(y[idx])) < 2: continue
        ms = [metrics(y[idx], p[idx]) for p in P]
        for k in KEYS: vals[k].append(np.mean([mm[k] for mm in ms]))
    return {k: np.percentile(v, [2.5, 97.5]) for k, v in vals.items()}
summ = []
for m in MODELS:
    for t in EVAL:
        d = per_seed[(per_seed.model == m) & (per_seed.test == t)]
        row = dict(model=m, test=t, n_seeds=len(d)); ci = boot_ci(m, t)
        for k in KEYS:
            row[f"{k}_mean"] = d[k].mean(); row[f"{k}_std"] = d[k].std(ddof=0) if len(d) > 1 else 0.0; row[f"{k}_ci_lo"], row[f"{k}_ci_hi"] = ci[k]
        if t == "test_own":
            for ft in ("t2i", "i2i"):
                dd = per_seed[(per_seed.model == m) & (per_seed.test == f"test_own/{ft}")]
                row[f"tpr_{ft}_mean"] = dd.tpr.mean(); row[f"tpr_{ft}_std"] = dd.tpr.std(ddof=0) if len(dd) > 1 else 0.0
        summ.append(row)
summary = pd.DataFrame(summ); summary.to_csv(f"{CFG['RES']}/metrics_summary.csv", index=False)
pd.set_option("display.width", 250)
show = summary.copy()
for k in ("acc", "auc", "ap", "tpr", "tnr"):
    show[k] = show.apply(lambda r: f"{r[k+'_mean']:.3f}±{r[k+'_std']:.3f} [{r[k+'_ci_lo']:.3f},{r[k+'_ci_hi']:.3f}]", axis=1)
print(show[["model", "test", "n_seeds", "acc", "auc", "ap", "tpr", "tnr"]].to_string(index=False))
print("\nTPR на test_own по подтипам (mean±std):")
print(show[show.test == "test_own"][["model", "tpr_t2i_mean", "tpr_t2i_std", "tpr_i2i_mean", "tpr_i2i_std"]].round(3).to_string(index=False))

## 7. Ранжирование vs порог: анализ калибровки на test_own
Для каждого детектора: accuracy при пороге 0,5; при пороге, максимизирующем balanced accuracy на val (CF, без доступа к целевому домену);
при пороге, подобранном на N_CAL=200 стратифицированных изображениях test_own и оценённом на остальных 1 800 (20 повторов);
и «oracle» — лучший возможный порог на всём test_own (верхняя граница).

In [ ]:
def best_thr(y, p):
    ts = np.unique(np.round(p, 3)); ba = [balanced_accuracy_score(y, (p >= t).astype(int)) for t in ts]
    return float(ts[int(np.argmax(ba))])
cal_rows = []
yo = EVAL["test_own"].y.values; yv = val_df.y.values
for m in MODELS:
    for s, d in PRED[m].items():
        p, pv = d["test_own"], d["val"]
        t_val = best_thr(yv, pv); t_orc = best_thr(yo, p)
        accs = []; rng2 = np.random.default_rng(s)
        for r in range(CFG["N_CAL_REPEATS"]):
            pos = rng2.choice(np.where(yo == 1)[0], CFG["N_CAL"] // 2, replace=False); neg = rng2.choice(np.where(yo == 0)[0], CFG["N_CAL"] // 2, replace=False)
            cal = np.concatenate([pos, neg]); rest = np.setdiff1d(np.arange(len(yo)), cal)
            t_cal = best_thr(yo[cal], p[cal]); accs.append(accuracy_score(yo[rest], (p[rest] >= t_cal).astype(int)))
        cal_rows.append(dict(model=m, seed=s, auc=roc_auc_score(yo, p),
            acc_thr05=accuracy_score(yo, (p >= 0.5).astype(int)), bacc_thr05=balanced_accuracy_score(yo, (p >= 0.5).astype(int)),
            thr_val=t_val, acc_thr_val=accuracy_score(yo, (p >= t_val).astype(int)),
            acc_thr_cal200_mean=np.mean(accs), acc_thr_cal200_std=np.std(accs),
            thr_oracle=t_orc, acc_thr_oracle=accuracy_score(yo, (p >= t_orc).astype(int)),
            median_p_real=float(np.median(p[yo == 0])), median_p_fake=float(np.median(p[yo == 1]))))
cal = pd.DataFrame(cal_rows); cal.to_csv(f"{CFG['RES']}/calibration_test_own.csv", index=False)
cal_summary = cal.groupby("model").agg(["mean", "std"]).drop(columns="seed"); cal_summary.to_csv(f"{CFG['RES']}/calibration_summary.csv")
print(cal.groupby("model")[["auc", "acc_thr05", "bacc_thr05", "acc_thr_val", "acc_thr_cal200_mean", "acc_thr_oracle", "thr_val", "thr_oracle", "median_p_real", "median_p_fake"]].mean().round(3).to_string())

## 8. По генераторам (среднее по seed'ам, порог 0,5) и графики

In [ ]:
rows = []
for m in MODELS:
    for s, d in PRED[m].items():
        for t, p in d.items():
            if t == "val": continue
            df = EVAL[t].copy(); df["p"] = p
            for (y, g), grp in df.groupby(["y", "group"]):
                rows.append(dict(model=m, seed=s, test=t, kind="fake" if y == 1 else "real", generator=g.replace("real:", ""),
                                 held_out=int(grp.held_out.max()), n=len(grp), rate=float((grp.p >= 0.5).mean()) if y == 1 else float((grp.p < 0.5).mean())))
G = pd.DataFrame(rows).drop_duplicates(["model", "seed", "generator", "kind", "test"])
Gs = G.groupby(["model", "test", "kind", "generator", "held_out", "n"]).rate.agg(["mean", "std"]).reset_index(); Gs.to_csv(f"{CFG['RES']}/per_generator.csv", index=False)
print(Gs[Gs.kind == "fake"].pivot_table(index=["test", "held_out", "generator"], columns="model", values="mean").round(3).to_string())

sets = ["test_seen", "test_unseen", "test_own"]; labels = ["seen\ngenerators", "unseen\ngenerators", "our set\n(COCO + SDXL-Turbo)"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, metric, title in zip(axes, ["auc", "acc"], ["ROC-AUC", "Accuracy (thr = 0.5)"]):
    w = 0.8 / len(MODELS)
    for i, m in enumerate(MODELS):
        r = summary[summary.model == m].set_index("test").loc[sets]
        mean = r[f"{metric}_mean"].values; err = np.vstack([mean - r[f"{metric}_ci_lo"].values, r[f"{metric}_ci_hi"].values - mean])
        ax.bar(np.arange(3) + i * w, mean, w, yerr=err, capsize=3, label=m)
        for j, v in enumerate(mean): ax.text(j + i * w, r[f"{metric}_ci_hi"].values[j] + 0.01, f"{v:.2f}", ha="center", fontsize=8)
    ax.set_xticks(np.arange(3) + w); ax.set_xticklabels(labels); ax.set_ylim(0.4, 1.08); ax.set_title(title + " (95% bootstrap CI)"); ax.grid(axis="y", alpha=0.3)
axes[0].legend(frameon=False, loc="lower left"); plt.tight_layout(); plt.savefig(f"{CFG['RES']}/figs/fig_main_v2.png", dpi=200); plt.close()

g = Gs[(Gs.kind == "fake") & (Gs.model == "CLIP+LR")].sort_values(["test", "mean"])
plt.figure(figsize=(9, 4.5)); colors = ["#4C72B0" if t == "test_seen" else "#DD8452" if t == "test_unseen" else "#55A868" for t in g.test]
plt.barh(g.generator + "  (" + g.test.str.replace("test_", "") + ")", g["mean"], color=colors)
plt.xlabel("fraction of fakes detected (thr = 0.5)"); plt.title("CLIP+LR: detection rate per generator"); plt.xlim(0, 1.05)
plt.tight_layout(); plt.savefig(f"{CFG['RES']}/figs/fig_per_generator_v2.png", dpi=200); plt.close()

# калибровка: accuracy на test_own при разных порогах
plt.figure(figsize=(6.5, 4)); ts = np.linspace(0.01, 0.99, 99)
for m in MODELS:
    p = np.mean([PRED[m][s]["test_own"] for s in PRED[m]], axis=0)
    plt.plot(ts, [accuracy_score(yo, (p >= t).astype(int)) for t in ts], label=m)
plt.axvline(0.5, color="gray", ls="--", lw=1); plt.xlabel("decision threshold"); plt.ylabel("accuracy on our set"); plt.ylim(0.4, 0.9)
plt.title("Accuracy on test-own as a function of the threshold"); plt.legend(frameon=False); plt.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f"{CFG['RES']}/figs/fig_calibration_v2.png", dpi=200); plt.close()
print("графики:", sorted(os.listdir(f"{CFG['RES']}/figs")))

## 9. Сохранение: predictions, config, архив → Drive

In [ ]:
P = []
for m in MODELS:
    for s, d in PRED[m].items():
        for t, p in d.items():
            df = (val_df if t == "val" else EVAL[t])[["id", "filename", "label", "origin", "generator", "fake_type", "source", "cluster"]].copy()
            df["model"], df["seed"], df["test"], df["prob_fake"] = m, s, t, p; P.append(df)
pd.concat(P).to_csv(f"{CFG['RES']}/predictions_all.csv", index=False)
json.dump({**{k: v for k, v in CFG.items()}, "device": DEV, "clip_C": bestC, "torch": torch.__version__}, open(f"{CFG['RES']}/config.json", "w"), indent=1, default=str)
with zipfile.ZipFile("/content/results_v2.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, fs in os.walk(CFG["RES"]):
        for f in fs:
            if not f.startswith("clip_feats"): z.write(os.path.join(root, f), os.path.relpath(os.path.join(root, f), CFG["RES"]))
try:
    shutil.copy("/content/results_v2.zip", "/content/drive/MyDrive/deepfake_paper/results_v2.zip"); print("скопировано: Drive / deepfake_paper / results_v2.zip")
except Exception as e:
    print("архив в /content/results_v2.zip", e)
from IPython.display import Image as IPImage, display
display(IPImage(f"{CFG['RES']}/figs/fig_main_v2.png")); display(IPImage(f"{CFG['RES']}/figs/fig_calibration_v2.png"))